# scatterplot — Python demo

Numerical companion to the entry [scatterplot](https://dictionaryofml.org/terms/scatterplot.html) of the [Dictionary of Applied Machine Learning](https://dictionaryofml.org/): it recomputes what the entry states and prints one line per check.

Numerical companion to the glossary entry 'scatterplot'. The 365 days of 2024 at Krems (GeoSphere station 3805) that carry both readings are data points. Each day carries the air temperature at 06:00 (UTC) as its feature and the temperature at 15:00 (UTC) of the same day as its label, so a scatterplot of the two puts one marker per day in the plane and a hypothesis that predicts the label from the feature is a curve in the same plane.

Requires NumPy and Matplotlib only, and uses fixed seeds, so the printed numbers reproduce exactly. Generated from [`pythondemos/scatterplot.py`](https://dictionaryofml.org/terms/scatterplot.py); CC BY 4.0.

Before running a downloaded copy, scan it: `uvx bandit -q -t B102,B307,B602,B605 scatterplot.py` (B102,B307,B602,B605 are the exec, eval and shell checks of [bandit](https://bandit.readthedocs.io/)) reports nothing: the script runs no shell command and evaluates no generated code.

In [ ]:
# Notebook shim: the script resolves output paths relative to __file__,
# which a notebook kernel does not define; everything lands in the
# working directory instead.
import os
__file__ = os.path.join(os.getcwd(), "scatterplot.py")
os.makedirs("pythondemos", exist_ok=True)

In [ ]:
"""The scatterplot of a dataset with two numeric features: the days of 2024
at Krems an der Donau drawn as markers, before and after a feature
transformation.

Purpose
-------
Numerical companion to the glossary entry 'scatterplot'.  The 365 days of
2024 at Krems (GeoSphere station 3805) that carry both readings are data
points.  Each day carries the air temperature at 06:00 (UTC) as its
feature and the temperature at 15:00 (UTC) of the same day as its label,
so a scatterplot of the two puts one marker per day in the plane and a
hypothesis that predicts the label from the feature is a curve in the
same plane.

The demo turns what a reader sees in such a picture into numbers that can
be checked: the elongated cloud is a sample correlation coefficient close
to one and a leading principal direction that carries almost all of the
sample variance; the day that sits far above the learned line is the
largest absolute residual; and the same days drawn in the transformed
features (the average of the two readings, the rise from the first to the
second) form a cloud whose spread is almost entirely along the first
axis, which is what a dimensionality reduction of these two features
exploits.

It also measures the house rule on how many markers a scatterplot may
carry.  Of the 365 days, many lie within half a degree of another day and
their markers overlap at any readable marker size.  Keeping every 20th
training day and every 15th validation and test day leaves 21 days, a
picture in which far fewer markers overlap, while the sample correlation
coefficient of the drawn days stays close to the 365-day value, so the
thinned picture still shows what the full record shows.

Self-contained (numpy and matplotlib only), fixed seed for the split; the
GeoSphere download falls back to the committed CSV.

Blocks
------
[B-data]      The 06:00 and the 15:00 (UTC) air temperature at Krems for
              2024 (365 days carry both readings).
[B-split]     A fixed-seed split of the 365 days into a training set, a
              validation set, and a test set.
[B-learn]     Least squares on the training set gives the linear
              hypothesis drawn as the solid line of panel (a).
[B-structure] The cloud of panel (a) as numbers: the sample correlation
              coefficient of the two features, the share of the sample
              variance along the leading principal direction, and the day
              with the largest absolute residual.
[B-transform] The same days in the transformed features (the average of
              the two readings, the rise between them): the sample
              correlation coefficient falls and the spread concentrates
              on the first axis.
[B-thin]      The deterministic thinning rule for the figure, 21 of the
              365 days: how many markers overlap before and after the
              thinning, and that the sample correlation coefficient
              survives it.
[B-plot]      CSVs for the two panels and the matplotlib preview.

Outputs
-------
scatterplot_weather.csv    : date, t06, t15 for 2024 (committed copy).
scatterplot_days_plot.csv  : the 21 drawn days with their two features,
                             their transformed features, and their split.
scatterplot_line.csv       : two points of the learned hypothesis.
scatterplot.png            : matplotlib preview (checking only).
"""

# /// script
# requires-python = ">=3.10"
# dependencies = ["matplotlib==3.10.7", "numpy==2.3.4"]
# ///

import json
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT_DIR = Path(__file__).parent

report = []


def check(name, ok):
    report.append((name, bool(ok)))
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")


def crowded(points, radius):
    """Number of points that lie within `radius` of another point."""
    diff = points[:, None, :] - points[None, :, :]
    near = np.sqrt((diff ** 2).sum(axis=2)) < radius
    np.fill_diagonal(near, False)
    return int(near.any(axis=1).sum())

**[B-data]** The 06:00 and the 15:00 (UTC) air temperature at Krems for 2024 (365 days carry both readings).

In [ ]:
print("[B-data] the 06:00 and the 15:00 air temperature at Krems, 2024, UTC")
URL = ("https://dataset.api.hub.geosphere.at/v1/station/historical/"
       "klima-v2-1h?parameters=tl&station_ids=3805"
       "&start=2024-01-01T00:00&end=2024-12-31T23:00")
CSV = OUT_DIR / "scatterplot_weather.csv"
try:
    with urllib.request.urlopen(URL, timeout=300) as resp:
        payload = json.load(resp)
    readings = payload["features"][0]["properties"]["parameters"]["tl"]["data"]
    byday = {}
    for stamp, value in zip(payload["timestamps"], readings):
        if value is None:                  # the archive drops whole days
            continue
        hour = stamp[11:16]                # the stamps are UTC, and so are
        if hour in ("06:00", "15:00"):     # the two times, so no conversion
            byday.setdefault(stamp[:10], {})[hour] = float(value)
    records = [(day, rec["06:00"], rec["15:00"])
               for day, rec in sorted(byday.items()) if len(rec) == 2]
    with open(CSV, "w") as fh:
        fh.write("date,t06,t15\n")
        for day, lo, hi in records:
            fh.write(f"{day},{lo},{hi}\n")
    print(f"    downloaded {len(records)} days with both readings")
except OSError as err:
    print(f"    download failed ({err}); reading the committed CSV")
    rows = CSV.read_text().strip().splitlines()[1:]
    records = [(d, float(a), float(b))
               for d, a, b in (r.split(",") for r in rows)]

check("[B-data] 365 of the days of 2024 carry both readings (the hourly "
      "record drops 2024-01-12)", len(records) == 365)
check("[B-data] no kept day is missing a reading",
      all(lo is not None and hi is not None for _, lo, hi in records))
dates = [d for d, _, _ in records]
x = np.array([lo for _, lo, _ in records], dtype=float)
y = np.array([hi for _, _, hi in records], dtype=float)
m = len(x)

**[B-split]** A fixed-seed split of the 365 days into a training set, a validation set, and a test set.

In [ ]:
print("[B-split] fixed-seed split of the 365 days")
rng = np.random.default_rng(2024)
perm = rng.permutation(m)
m_train, m_val = 219, 73
split = np.empty(m, dtype=int)                 # 0 train, 1 val, 2 test
split[perm[:m_train]] = 0
split[perm[m_train:m_train + m_val]] = 1
split[perm[m_train + m_val:]] = 2
counts = [int((split == s).sum()) for s in (0, 1, 2)]
check(f"[B-split] {counts[0]} training days, {counts[1]} validation days, "
      f"{counts[2]} test days, {sum(counts)} in total",
      counts == [219, 73, 73] and sum(counts) == m)

**[B-learn]** Least squares on the training set gives the linear hypothesis drawn as the solid line of panel (a).

In [ ]:
print("[B-learn] linear hypothesis learned on the training days")
tr = split == 0
A = np.column_stack([np.ones(int(tr.sum())), x[tr]])
w0, w1 = np.linalg.lstsq(A, y[tr], rcond=None)[0]


def h(xx):
    return w0 + w1 * xx


res = y - h(x)
trainerr = float(np.mean(res[tr] ** 2))
valerr = float(np.mean(res[split == 1] ** 2))
check(f"[B-learn] h(x) = {w0:.2f} + {w1:.3f} x; the average squared "
      f"deviation of a day from this line is {trainerr:.1f} on the training "
      f"days and {valerr:.1f} on the validation days",
      0.5 < w1 < 1.5 and trainerr > 0)

**[B-structure]** The cloud of panel (a) as numbers: the sample correlation coefficient of the two features, the share of the sample variance along the leading principal direction, and the day with the largest absolute residual.

In [ ]:
print("[B-structure] the elongated cloud as numbers")
rho = float(np.corrcoef(x, y)[0, 1])
check(f"[B-structure] sample correlation coefficient of the two features "
      f"{rho:.3f}: the markers of the 365 days lie close to a rising line",
      rho > 0.8)
X = np.column_stack([x, y])
ev = np.sort(np.linalg.eigvalsh(np.cov(X, rowvar=False)))[::-1]
share = float(ev[0] / ev.sum())
check(f"[B-structure] the leading principal direction carries "
      f"{100 * share:.1f} percent of the sample variance of the two "
      f"features", share > 0.9)
far = int(np.argmax(np.abs(res)))
check(f"[B-structure] the largest absolute residual is {abs(res[far]):.1f} "
      f"degrees, on {dates[far]} (06:00 reading {x[far]:.1f}, 15:00 "
      f"reading {y[far]:.1f}, prediction {h(x[far]):.1f})",
      abs(res[far]) > 3 * np.sqrt(trainerr) / 2)

**[B-transform]** The same days in the transformed features (the average of the two readings, the rise between them): the sample correlation coefficient falls and the spread concentrates on the first axis.

In [ ]:
print("[B-transform] the average of the two readings and the rise")
u = (x + y) / 2.0                              # average of the two readings
v = y - x                                      # rise from 06:00 to 15:00
rho_t = float(np.corrcoef(u, v)[0, 1])
check(f"[B-transform] sample correlation coefficient of the transformed "
      f"features {rho_t:.3f}, against {rho:.3f} for the recorded ones",
      abs(rho_t) < abs(rho))
share_rec = float(np.var(x) / (np.var(x) + np.var(y)))
share_u = float(np.var(u) / (np.var(u) + np.var(v)))
check(f"[B-transform] the average of the two readings carries "
      f"{100 * share_u:.1f} percent of the sum of the two sample variances, "
      f"the rise {100 * (1 - share_u):.1f} percent; the recorded "
      f"features split the same sum {100 * share_rec:.1f} to "
      f"{100 * (1 - share_rec):.1f}", share_u > share_rec)
check("[B-transform] the transformation is invertible: the two readings "
      "are recovered as u - v/2 and u + v/2",
      np.allclose(u - v / 2, x) and np.allclose(u + v / 2, y))

**[B-thin]** The deterministic thinning rule for the figure, 21 of the 365 days: how many markers overlap before and after the thinning, and that the sample correlation coefficient survives it.

In [ ]:
print("[B-thin] every 20th training day and every 15th validation and "
      "test day")
STRIDE = (20, 15, 15)
idx = np.sort(np.concatenate(
    [np.flatnonzero(split == s)[::STRIDE[s]] for s in (0, 1, 2)]))
drawn = [int((split[idx] == s).sum()) for s in (0, 1, 2)]
check(f"[B-thin] every 20th of the {counts[0]} training days, every 15th of "
      f"the {counts[1]} validation days, and every 15th of the {counts[2]} "
      f"test days: {drawn[0]} + {drawn[1]} + {drawn[2]} = {len(idx)} of the "
      f"{m} days are drawn", drawn == [11, 5, 5] and len(idx) == 21)
RADIUS = 0.5                                   # degrees
crowd_all = crowded(X, RADIUS)
crowd_thin = crowded(X[idx], RADIUS)
check(f"[B-thin] {crowd_all} of the {m} days lie within {RADIUS} degrees of "
      f"another day, so their markers overlap at any readable marker size; "
      f"{crowd_thin} of the {len(idx)} drawn days do", crowd_thin < crowd_all)
rho_thin = float(np.corrcoef(x[idx], y[idx])[0, 1])
check(f"[B-thin] sample correlation coefficient of the drawn days "
      f"{rho_thin:.3f}, within {abs(rho_thin - rho):.3f} of the {m}-day "
      f"value {rho:.3f}", abs(rho_thin - rho) < 0.1)

**[B-plot]** CSVs for the two panels and the matplotlib preview.

In [ ]:
print("[B-plot] CSVs for the two panels and the preview")
with open(OUT_DIR / "scatterplot_days_plot.csv", "w") as fh:
    fh.write("t06,t15,tmean,trise,split\n")
    for i in idx:
        fh.write(f"{x[i]:.1f},{y[i]:.1f},{u[i]:.2f},{v[i]:.2f},"
                 f"{split[i]}\n")
x_lo, x_hi = float(x.min()), float(x.max())
with open(OUT_DIR / "scatterplot_line.csv", "w") as fh:
    fh.write("t06,pred\n")
    for xx in (x_lo, x_hi):
        fh.write(f"{xx:.2f},{h(xx):.3f}\n")

MARKS = ((0, "o", "k", "training set"),
         (1, "^", "none", "validation set"),
         (2, "s", "none", "test set"))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.4, 3.9))
for s, mk, fc, lab in MARKS:
    sel = idx[split[idx] == s]
    ax1.scatter(x[sel], y[sel], marker=mk, facecolors=fc,
                edgecolors="black", s=28, label=lab)
    ax2.scatter(u[sel], v[sel], marker=mk, facecolors=fc,
                edgecolors="black", s=28, label=lab)
ax1.plot([x_lo, x_hi], [h(x_lo), h(x_hi)], "k-", lw=0.9,
         label="learned hypothesis")
ax1.set_xlabel("06:00 (UTC) reading (°C)")
ax1.set_ylabel("15:00 (UTC) reading (°C)")
ax1.set_title("(a) the two recorded features")
ax1.legend(frameon=False, fontsize=8)
ax2.set_xlabel("average of the two readings (°C)")
ax2.set_ylabel("rise from 06:00 to 15:00 (°C)")
ax2.set_title("(b) the two transformed features")
ax2.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(OUT_DIR / "scatterplot.png", dpi=110)

n_ok = sum(ok for _, ok in report)
print(f"\n{n_ok}/{len(report)} checks pass")
if n_ok != len(report):
    raise SystemExit(1)